# Classification Multi-Label des Pathologies Thoraciques sur CheXpert (Small)

## Résumé Exécutif

Dans ce notebook, nous traitons un problème de classification multi-label
pour la détection automatique de pathologies thoraciques à partir d’images
de radiographies pulmonaires (Chest X-Ray). Notre objectif est d'entrainner un modèle capable de prendre en charge les **dépendances hiérarchiques** et l'**étiquette d'incertitude**.

Nous utilisons le jeu de données **CheXpert (version small, ~11 Go)**,
disponible sur Kaggle.  
Chaque image peut contenir plusieurs pathologies simultanément,
ce qui en fait un problème de classification multi-étiquette.

---

## Stratégie d’Entraînement

Nous adoptons une stratégie d’apprentissage en deux étapes :

### 1️⃣ Étape 1 – Apprentissage global
Le modèle est entraîné pendant **2 époques** sur l’ensemble des pathologies disponibles.
L’objectif est d’apprendre des représentations générales des anomalies thoraciques.

### 2️⃣ Étape 2 – Fine-tuning ciblé
Lors de la **3e époque**, nous effectuons un affinement (fine-tuning)
du modèle sur **5 pathologies sélectionnées**.

Ces pathologies ont été choisies selon :
- Leur intérêt clinique
- Leur prévalence dans le jeu de données
- Leur importance diagnostique

Cette stratégie vise à spécialiser le modèle sur des maladies prioritaires
tout en conservant les représentations apprises globalement.

---

## Gestion des Étiquettes d’Incertitude

Le jeu de données CheXpert inclut des étiquettes incertaines (notées "U").
Dans ce travail, nous adoptons une stratégie dite **U-Ones** :

Les étiquettes incertaines sont remplacées par 1,
c’est-à-dire considérées comme positives.

Cette décision est motivée par une logique de dépistage médical :
en cas de doute, il est préférable de considérer la pathologie
comme potentiellement présente afin de réduire le risque de faux négatifs.

---

## Objectifs

Les objectifs de ce travail sont :

- Entraîner un modèle robuste de classification multi-label
- Étudier l’apport du fine-tuning ciblé

## Import

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from torch.cuda.amp import GradScaler, autocast

from sklearn.metrics import roc_auc_score
import torch.nn.functional as F

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

## Configurations

In [ ]:
# === CONFIGURATION ===
CSV_PATH = "/kaggle/input/chexpert-v10-small/CheXpert-v1.0-small/train.csv"
IMG_DIR = "/kaggle/input/chexpert-v10-small/"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Labels cibles
TARGET_5 = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]
ALL_LABELS = [
    "No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity", "Lung Lesion",
    "Edema", "Consolidation", "Pneumonia", "Atelectasis", "Pneumothorax",
    "Pleural Effusion", "Pleural Other", "Fracture", "Support Devices"
]

## Dataset

## 1. Présentation du Dataset : CheXpert (Small)

Nous utilisons le jeu de données **CheXpert (version small)**,
disponible sur Kaggle.

CheXpert est un large dataset de radiographies thoraciques
annotées automatiquement à partir de rapports radiologiques.

---

### Source

Le dataset original a été publié par l'Université de Stanford :

Irvin et al., *CheXpert: A Large Chest Radiograph Dataset with Uncertainty Labels and Expert Comparison*, AAAI 2019.

La version "small" utilisée ici (~11 Go) constitue un sous-ensemble
permettant des expérimentations plus rapides.

---

### Nature des données

- Images : radiographies thoraciques (Chest X-Ray)
- Modalité : projection frontale (principalement)
- Type de tâche : classification multi-label
- Format des labels : CSV

Chaque image peut contenir **plusieurs pathologies simultanément**.

---

### Pathologies considérées

CheXpert inclut 14 observations, dont :

- Cardiomegaly
- Edema
- Consolidation
- Atelectasis
- Pleural Effusion
- Pneumonia
- Pneumothorax
- Enlarged Cardiostinum
- Fracture
- Lung Opacity
- Lung Lesion
- Pleural Other
- No Finding
- Support device

Dans ce travail :

- Étape 1 : entraînement sur l'ensemble des pathologies
- Étape 2 : fine-tuning ciblé sur 5 pathologies sélectionnées
  pour leur prévalence et leur intérêt clinique

---

### Système d'annotation

Chaque pathologie peut prendre 4 valeurs :

- 1 : Présente
- 0 : Absente
- -1 : Incertaine
- NaN : Non mentionnée

---

### Gestion des incertitudes

Nous adoptons une stratégie **U-Ones** :

- Les labels -1 (incertains) sont remplacés par 1.

Motivation :
Dans un contexte de dépistage médical, il est préférable
de considérer une pathologie incertaine comme potentiellement présente
afin de limiter les faux négatifs.

Cette décision influence la distribution des classes
et peut augmenter la sensibilité au détriment de la spécificité.

---

### Déséquilibre des classes

Comme souvent en imagerie médicale,
le dataset présente un fort déséquilibre :

- Certaines pathologies sont très rares
- D'autres sont fortement représentées

Ce déséquilibre justifie :
- L'utilisation d'une loss adaptée (BCE)
- Une analyse des performances par pathologie

### Transformations et Data Augmentation

Les transformations suivantes sont appliquées :

1. **Resize (224 × 224)**  
   Adaptation à l'entrée standard des architectures pré-entraînées
   comme ResNet (ImageNet).

2. **Random Horizontal Flip (p = 0.5)**  
   Augmentation des données afin d'améliorer la robustesse
   du modèle aux variations spatiales.

3. **Random Rotation (±10 degrés)**  
   Simulation de légères variations d’angle
   observées en pratique clinique.

4. **Conversion en tenseur**  
   Transformation de l’image en tenseur PyTorch.

5. **Normalisation (ImageNet mean/std)**  
   Moyennes : [0.485, 0.456, 0.406]  
   Écarts-types : [0.229, 0.224, 0.225]

   Cette normalisation est cohérente avec
   un modèle pré-entraîné sur ImageNet,
   garantissant une distribution d’entrée compatible.



In [ ]:
# === DATASET ===
class CheXpertDataset(Dataset):
    def __init__(self, df, image_root, labels, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_root = image_root
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_root, row["Path"])
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        label = row[self.labels].fillna(0).values.astype(np.float32)
        label[label == -1] = 1.0

        return image, torch.tensor(label)

# === TRANSFORMATIONS ===
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# === LOAD DATA ===
df = pd.read_csv(CSV_PATH)
df.fillna(0, inplace=True)
df = df[df["Frontal/Lateral"] == "Frontal"]

## Model

## 2. Formulation Mathématique du Problème

Nous considérons un problème de **classification multi-label**.

Contrairement à la classification multi-classe (où une seule classe est vraie),
une radiographie peut présenter plusieurs pathologies simultanément.

---

### Définition des variables

Soit :

- $X \in \mathbb{R}^{H \times W}$ une image radiographique
- $Y \in \{0,1\}^K$ un vecteur multi-étiquette
- $K$ le nombre total de pathologies

Chaque composante $Y_k$ indique la présence (1) ou l'absence (0)
de la pathologie $k$.

Nous cherchons à apprendre une fonction :

$$
f_\theta : X \rightarrow \hat{Y}
$$

où :

- $f_\theta$ est un réseau de neurones paramétré par $\theta$
- $\hat{Y} \in [0,1]^K$ représente les probabilités prédites

---

### Activation Sigmoïde

Dans un problème multi-label, les sorties ne sont pas exclusives.
Nous utilisons donc une activation sigmoïde indépendante pour chaque sortie :

$$
\hat{y}_k = \sigma(z_k) = \frac{1}{1 + e^{-z_k}}
$$

Contrairement au softmax, chaque pathologie est prédite indépendamment.

---

### Fonction de perte

Nous utilisons la Binary Cross-Entropy avec logits (BCEWithLogitsLoss) :

$$
\mathcal{L} = - \frac{1}{K} \sum_{k=1}^{K}
\left[
y_k \log(\hat{y}_k)
+
(1 - y_k) \log(1 - \hat{y}_k)
\right]
$$

Cette perte est adaptée aux problèmes multi-label
car elle traite chaque pathologie indépendamment.

---

### Seuil de décision

Pour obtenir des prédictions binaires :

$$
\hat{y}_k =
\begin{cases}
1 & \text{si } \hat{y}_k \ge \tau \\
0 & \text{sinon}
\end{cases}
$$

Dans ce travail, nous utilisons initialement $\tau = 0.5$.

In [ ]:
# === MODEL ===
def create_model(num_classes):
    model = models.densenet121(weights=True)
    model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    return model.to(DEVICE)

# === TRAIN FUNCTION ===
def train_epoch(loader, model, optimizer, criterion, scaler):
    model.train()
    total_loss = 0
    for images, labels in tqdm(loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)

## Entrainement

In [1]:
# === PHASE 1 : Entraînement sur les 14 pathologies ===
print("\n=== PHASE 1 : 14 pathologies ===")
model = create_model(num_classes=len(ALL_LABELS))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)
#scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.1)
criterion = nn.BCEWithLogitsLoss()
scaler = GradScaler()

dataset14 = CheXpertDataset(df, IMG_DIR, ALL_LABELS, transform)
loader14 = DataLoader(dataset14, batch_size=16, shuffle=True, num_workers=2)

for epoch in range(3):
    loss = train_epoch(loader14, model, optimizer, criterion, scaler)
    print(f"[14C] Époque {epoch+1}/3 - Loss: {loss:.4f}")
    scheduler.step(loss)  # 🔁 déclencheur du scheduler

# === PHASE 2 : Fine-tuning sur les 5 pathologies ===
print("\n=== PHASE 2 : 5 pathologies ===")

# Remplacer la tête
model.classifier = nn.Linear(model.classifier.in_features, len(TARGET_5)).to(DEVICE)

# Gele les 3 premiers dense
for name, param in model.features.named_parameters():
    if (
        'denseblock1' in name or
        'transition1' in name or
        'denseblock2' in name or
        'transition2' in name or
        'denseblock3' in name
    ):
        param.requires_grad = False

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5  # ou plus petit si tu veux un fine-tuning fin (ex : 1e-5)
)
#optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)
#scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.1)
criterion = nn.BCEWithLogitsLoss()
scaler = GradScaler()

dataset5 = CheXpertDataset(df, IMG_DIR, TARGET_5, transform)
loader5 = DataLoader(dataset5, batch_size=16, shuffle=True, num_workers=2)

for epoch in range(2):
    loss = train_epoch(loader5, model, optimizer, criterion, scaler)
    print(f"[5C] Époque {epoch+1}/2 - Loss: {loss:.4f}")
    scheduler.step(loss)  # 🔁 déclencheur du scheduler

# Optionnel : sauvegarde du modèle
torch.save(model.state_dict(), "chexpert_5_pathologies_finetuned.pth")

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth



=== PHASE 1 : 14 pathologies ===


100%|██████████| 30.8M/30.8M [00:00<00:00, 188MB/s]
100%|██████████| 11940/11940 [23:40<00:00,  8.40it/s]


[14C] Époque 1/3 - Loss: 0.3620


100%|██████████| 11940/11940 [23:11<00:00,  8.58it/s]


[14C] Époque 2/3 - Loss: 0.3471


100%|██████████| 11940/11940 [18:56<00:00, 10.51it/s]


[5C] Époque 1/2 - Loss: 0.4533


100%|██████████| 11940/11940 [18:43<00:00, 10.63it/s]


[5C] Époque 2/2 - Loss: 0.4412


## Evaluation

## 3. Jeu de Validation et Pathologies Ciblées

### Jeu de Validation

Le jeu de validation fourni avec CheXpert est annoté avec un soin particulier
et sert de référence pour l'évaluation des performances.

Il permet :

- Une évaluation indépendante du modèle
- Une comparaison objective entre différentes stratégies
- Une analyse fine par pathologie

Les performances sont mesurées séparément pour chaque maladie,
ce qui est essentiel en contexte multi-label médical.

---

### Pathologies Sélectionnées pour le Fine-Tuning

Lors de la seconde phase d'entraînement,
nous focalisons le modèle sur 5 pathologies :

- Cardiomegaly
- Edema
- Consolidation
- Atelectasis
- Pleural Effusion

Ces pathologies ont été choisies pour :

- Leur importance clinique
- Leur fréquence suffisante dans le dataset
- Leur intérêt diagnostique en pratique hospitalière

Le jeu de validation contient également ces 5 pathologies,
ce qui permet une évaluation cohérente du fine-tuning ciblé.

---

In [ ]:
# === CHARGER valid.csv ===
valid_df = pd.read_csv("/kaggle/input/chexpert-v10-small/CheXpert-v1.0-small/valid.csv")
valid_df = valid_df[valid_df["Frontal/Lateral"] == "Frontal"]
valid_df.fillna(0, inplace=True)

# === Dataset de validation ===
val_dataset = CheXpertDataset(
    df=valid_df,
    image_root="/kaggle/input/chexpert-v10-small/",
    labels=TARGET_5,
    transform=transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    #allow_uncertainty=False  # On ne veut pas toucher aux labels du valid set !
)

val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

In [2]:
# === Fonction d’évaluation ===
def evaluate_auc(model, loader, labels):
    model.eval()
    all_targets = []
    all_outputs = []

    with torch.no_grad():
        for images, targets in tqdm(loader):
            images = images.to(DEVICE)
            outputs = model(images)
            outputs = torch.sigmoid(outputs).cpu().numpy()
            targets = targets.cpu().numpy()
            all_outputs.append(outputs)
            all_targets.append(targets)

    y_true = np.concatenate(all_targets, axis=0)
    y_pred = np.concatenate(all_outputs, axis=0)

    aucs = {}
    for i, label in enumerate(labels):
        try:
            auc = roc_auc_score(y_true[:, i], y_pred[:, i])
        except ValueError:
            auc = float('nan')  # Pas assez de classes positives/négatives
        aucs[label] = auc

    return aucs

# === Charger le modèle fine-tuné (si nécessaire) ===
# model.load_state_dict(torch.load("chexpert_5_pathologies_finetuned.pth"))

# === Évaluer ===
aucs = evaluate_auc(model, val_loader, TARGET_5)
print("\n🎯 AUC par pathologie :")
for label, auc in aucs.items():
    print(f"{label:<20} : {auc:.4f}")


100%|██████████| 13/13 [00:02<00:00,  5.14it/s]


🎯 AUC par pathologie :
Atelectasis          : 0.8156
Cardiomegaly         : 0.8367
Consolidation        : 0.8640
Edema                : 0.9379
Pleural Effusion     : 0.9291



## 4. Comparaison de Deux Stratégies de Classification

Nous comparons quatre approches :

1. **U-Ones5**
   - Selectionner uniquement nos 5 pathologies
   - Et remplacer -1 par 1 pour l'étiquette d'incertitude.


2. **U-Ones14**
   - Et remplacer -1 par 1 pour l'étiquette d'incertitude.

     
3. **Pham et al, 2021**
    - **Apprentissage conditionnel** : Le CNN est pré-entraîné sur un sous-ensemble où les nœuds parents sont positifs pour capturer **les dépendances hiérarchiques**. Cela force le modèle à mieux distinguer les pathologies de bas niveau, dites "nœuds feuilles".
    - **Affinage (Fine-tuning)** : Le modèle est ajusté sur l'ensemble des données en gelant toutes les couches sauf la dernière, qui est entièrement connectée. Ce processus vise spécifiquement à améliorer la précision des prédictions pour les étiquettes parentales.


4. **Notre méthode**
   - Pré-entraîné sur toutes les pathologies
   - Affiné spécifiquement sur les 5 pathologies sélectionnées
---

### Tableau Comparatif (exemple illustratif)

| Méthode                | Cardiomegaly (AUC) | Edema (AUC) | Consolidation (AUC) | Atelectasis (AUC) | Pleural Effusion (AUC) | Moyenne |
|------------------------|--------------------|-------------|---------------------|-------------------|------------------------|---------|
| U-Ones5          | 0.74               | 0.92        | 0.89                | 0.83              | 0.91                   | 0.858    |
| U-Ones14       | 0.74               | **0.93**        | 0.89                | 0.81              | 0.91                   | 0.856    |
| Pham et al          | 0.45               | 0.57        | 0.62                | 0.64              | 0.89                   | 0.640    |
| Notre méthode       | **0.79**               | 0.92        | **0.92**                | 0.82              | 0.91                   | **0.868**    |

---

### Discussions

- **U-Ones14 vs U-Ones5**
**U-Ones5** obtient de meilleures performances que \textbf{U-Ones14}. Cela s’explique par sa spécialisation : il est spécifiquement entraîné pour détecter les cinq pathologies utilisées dans l’évaluation, ce qui le rend plus adapté à la tâche cible.

- **U-Ones5 vs Méthode proposée**
Notre méthode surpasse **U-Ones5**. Bien que ce dernier soit spécialisé sur les cinq pathologies d’évaluation, notre approche bénéficie d’une phase d’entraînement initiale sur l’ensemble des étiquettes, ce qui lui confère une meilleure capacité de généralisation ( désigne la capacité d’un modèle à bien se comporter sur des données qu’il n’a jamais vues lors de son entraînement), tout en conservant une spécialisation lors du fine-tuning.

- **U-Ones14 vs Méthode proposée**
Notre méthode améliore les performances de **U-Ones14** grâce à une étape de fine-tuning ciblée. En affinant ce modèle généraliste sur les cinq pathologies d’intérêt, on combine la richesse de l’apprentissage global avec une spécialisation pertinente.

- **Pham et al. (2021) vs Méthode proposée**
Notre approche surpasse celle de **Pham et al. (2021)** pour les raisons évoquées dans les comparaisons précédentes : une exposition complète aux données, et un fine-tuning ciblé, contrairement à un apprentissage où  la partie convolutive n’est jamais exposée à l’ensemble du jeu de données avec des étiquettes d'incertitude modifiées aléatoirement.

## Références

[1] Jeremy Irvin et al.  
*CheXpert: A Large Chest Radiograph Dataset with Uncertainty Labels and Expert Comparison*  
AAAI Conference on Artificial Intelligence, 2019.

[2] Pham et al.  
*Interpreting chest X-rays via CNNs that exploit hierarchical disease dependencies and uncertainty labels* 
